# Hotel Booking Demand: Do early bookings cancel more?
Vikram Manisankar · Advanced Business Statistics · A1 Individual Assignment

## Business problem

**Stakeholder:** the Revenue Manager of the City Hotel.

**Decision:** whether to make cancellation rules stricter (for example a small deposit, or free cancellation only until 30 days before arrival) for bookings made more than 90 days in advance.

**Research question:** Are City Hotel bookings made more than 90 days before arrival cancelled more often than bookings made 90 days or less before arrival, and is the gap big enough to matter?

**Why it matters:** a cancelled booking is a room the hotel held but did not sell. If early bookings cancel much more, the hotel carries a lot of risk on rooms it holds months ahead.

**What counts as "big enough":** a stricter rule could also turn some guests away, so a small gap would not justify it. Before running the test, I set **10 percentage points** as the minimum gap worth acting on.

**Variables I use:**
- `is_canceled`: 1 if cancelled, 0 if not (categorical)
- `lead_time`: days between booking and arrival (numerical)
- `hotel`: City Hotel or Resort Hotel (categorical)
- `adr`: average daily rate, the price per night in \$ (numerical)
- `stays_in_weekend_nights`, `stays_in_week_nights`: length of stay (numerical)
- `market_segment`, `deposit_type`: used to check other explanations (categorical)

**Hypotheses (α = 0.05):**
- H₀: early bookings (lead time > 90 days) and late bookings (≤ 90 days) have the same cancellation rate.
- H₁: early bookings have a higher cancellation rate than late bookings.

The data is observational, so I can show associations but not causes.

## 1. Load and inspect the data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
from scipy import stats
from google.colab import files

When the next cell runs, click **Choose Files** and upload `hotel_booking_demand.csv` (the dataset from Canvas, with that exact file name).

In [ ]:
uploaded = files.upload()

df = pd.read_csv("hotel_booking_demand.csv")

In [ ]:
df.head()

Each row is **one hotel booking**.

In [ ]:
df.shape

In [ ]:
df.info()

**Data types:** `children` is stored as a float only because of 4 missing values, so I convert it to an integer after filling them. `agent` and `company` are ID numbers, not real quantities, and I do not use them. The other columns I use already have the right type: text for categories, numbers for lead time, price and nights.

## 2. Data quality checks

### Missing values

In [ ]:
df.isnull().sum()

`company` and `agent` are mostly empty because most bookings did not go through a company or an agent, so empty simply means "none". Only `children` (4 rows) and `country` (488 rows) have real gaps, which I fill (no children and "Unknown").

### Duplicates

In [ ]:
# Count rows that are exactly the same
duplicates = df.duplicated().sum()
duplicates

In [ ]:
duplicates / len(df)

27% of the rows are exact duplicates. With no booking ID, these could be real bookings (for example a group reserving several identical rooms), so I keep them and check later whether removing them changes the result.

### Values that do not make sense

In [ ]:
# New variables: total nights and total guests
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"].fillna(0) + df["babies"]

In [ ]:
print("Bookings with 0 guests:", (df["total_guests"] == 0).sum())
print("Bookings with 0 nights:", (df["total_nights"] == 0).sum())
print("Negative ADR:", (df["adr"] < 0).sum())
print("Undefined market segment:", (df["market_segment"] == "Undefined").sum())

In [ ]:
# is_canceled should only be 0 or 1
df["is_canceled"].unique()

In [ ]:
# Bookings with children but no adults
((df["adults"] == 0) & (df["children"] > 0)).sum()

223 bookings have children but no adults. This is unusual but possible (for example, the adults were not entered), and it is a small number, so I keep them.

In [ ]:
print("ADR range:", df["adr"].min(), "-", df["adr"].max())

In [ ]:
fig = px.box(df, x="hotel", y="adr", title="Price per Night by Hotel (before cleaning)",
             labels={"hotel": "Hotel", "adr": "Average daily rate ($ per night)"})
fig.show()

One booking is \$5,400 per night, while the next highest is \$510, so it is almost certainly a typo. The longest lead time is 737 days, which is long but realistic for a hotel booking, so I keep it.

## 3. Clean the data

What I removed and why:
- 180 bookings with 0 guests, because a booking with nobody staying is not a real stay.
- 1 booking with a negative price, because a price cannot be below \$0.
- 1 booking at \$5,400 per night, because it is most likely an error.
- 2 bookings with an "Undefined" market segment, because I cannot interpret them.

What I kept: bookings with \$0 price or 0 nights are still real bookings, so I keep them for the cancellation question. I only leave them out when I calculate revenue.

In [ ]:
rows_before = len(df)

In [ ]:
df = df[df["total_guests"] > 0]

In [ ]:
df = df[(df["adr"] >= 0) & (df["adr"] < 1000)]

In [ ]:
df = df[df["market_segment"] != "Undefined"]

In [ ]:
# Fill the few missing values
df["children"] = df["children"].fillna(0).astype(int)
df["country"] = df["country"].fillna("Unknown")

In [ ]:
print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(df))

In [ ]:
fig = px.box(df, x="hotel", y="adr", title="Price per Night by Hotel (after cleaning)",
             labels={"hotel": "Hotel", "adr": "Average daily rate ($ per night)"})
fig.show()

After removing the \$5,400 error, the prices are easy to compare. At the City Hotel the middle 50% of bookings cost about \$79 to \$126 per night (median \$100). The Resort Hotel is cheaper (median \$75) but varies more, from about \$50 to \$125, probably because resort prices change more between seasons.

### Create new variables
- `lead_group`: **Early** if the booking was made more than 90 days before arrival, **Late** if 90 days or less. 90 days is about one quarter ahead, a practical cutoff I picked before testing.
- `lead_band`: smaller lead-time groups, to see if the pattern is gradual.
- `booking_revenue`: the price per night times the number of nights, the room revenue of the booking.

In [ ]:
df["lead_group"] = np.where(df["lead_time"] > 90, "Early", "Late")

In [ ]:
df["lead_band"] = pd.cut(df["lead_time"], bins=[-1, 7, 30, 90, 180, 365, 800],
                         labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"])

In [ ]:
df["booking_revenue"] = df["adr"] * df["total_nights"]

In [ ]:
# My analysis focuses on the City Hotel
city = df[df["hotel"] == "City Hotel"]
len(city)

In [ ]:
# Share of City Hotel bookings that are early (more than 90 days ahead)
(city["lead_group"] == "Early").mean().round(3)

## 4. Descriptive statistics

In [ ]:
# Cancellation rate in each hotel
df.groupby("hotel")["is_canceled"].mean().round(3)

In [ ]:
# Lead time at the City Hotel
city["lead_time"].describe().round(1)

In [ ]:
# Number of bookings and cancellation rate for early and late bookings
city.groupby("lead_group")["is_canceled"].agg(["count", "mean"]).round(3)

In [ ]:
# Average price per night for early and late bookings
city.groupby("lead_group")["adr"].mean().round(2)

- The City Hotel cancels far more than the Resort Hotel (42% vs 28%).
- Lead time is right-skewed (mean 110 days, median 74), so the median is the better "typical" value.
- 56% of early bookings were cancelled vs 31% of late ones, while the price per night is almost the same. The difference is about cancellations, not price.

## 5. Visualizations

### 5.1 Distribution of lead time

In [ ]:
fig = px.histogram(city, x="lead_time", nbins=50, title="Distribution of Lead Time (City Hotel)",
                   labels={"lead_time": "Lead time (days before arrival)"})
fig.add_vline(x=city["lead_time"].median(), line_dash="dash", line_color="red",
              annotation_text="Median", annotation_position="top right")
fig.update_layout(yaxis_title="Number of bookings")
fig.show()

Most bookings are made within a few months of arrival, but there is a long tail out to a year or more. Early bookings are still 44% of the total, so they matter for the hotel.

### 5.2 Cancellation rate by lead time

In [ ]:
band_rates = df.groupby(["hotel", "lead_band"], observed=True)["is_canceled"].mean().reset_index()
band_rates["cancellation_pct"] = band_rates["is_canceled"] * 100
band_rates

In [ ]:
fig = px.bar(band_rates, x="lead_band", y="cancellation_pct", color="hotel", barmode="group",
             text_auto=".0f", title="Cancellation Rate by Lead Time",
             labels={"lead_band": "Lead time (days)", "cancellation_pct": "Cancellation rate (%)", "hotel": "Hotel"})
fig.show()

In both hotels, cancellations rise steadily the further ahead people book. At the City Hotel they go from 12% (booked within a week) to 71% (booked more than a year ahead). The rise is gradual rather than a jump at 90 days, and it is much steeper at the City Hotel, which is why I focus there.

### 5.3 Confidence intervals for the cancellation rate

A 95% confidence interval gives a range of plausible values for the true cancellation rate. I build it the same way as in class: the sample proportion, plus or minus 1.96 standard errors.

In [ ]:
# Early bookings
early = city[city["lead_group"] == "Early"]["is_canceled"]
n_early = len(early)
p_early = early.mean()
se_early = np.sqrt(p_early * (1 - p_early) / n_early)
L_early, U_early = p_early - 1.96 * se_early, p_early + 1.96 * se_early
n_early, p_early, L_early, U_early

In [ ]:
# Late bookings
late = city[city["lead_group"] == "Late"]["is_canceled"]
n_late = len(late)
p_late = late.mean()
se_late = np.sqrt(p_late * (1 - p_late) / n_late)
L_late, U_late = p_late - 1.96 * se_late, p_late + 1.96 * se_late
n_late, p_late, L_late, U_late

In [ ]:
# Table of results (in %)
res = pd.DataFrame({"lead_group": ["Early", "Late"],
                    "cancellation_pct": [p_early * 100, p_late * 100],
                    "L": [L_early * 100, L_late * 100],
                    "U": [U_early * 100, U_late * 100]})
res["error"] = res["U"] - res["cancellation_pct"]
res

In [ ]:
fig = px.scatter(res, x="lead_group", y="cancellation_pct", error_y="error", color="lead_group",
                 title="City Hotel Cancellation Rate with 95% Confidence Intervals",
                 labels={"lead_group": "Booking group", "cancellation_pct": "Cancellation rate (%)"})
fig.update_layout(yaxis_range=[0, 70])
fig.show()

Both intervals are very narrow (35,143 early and 44,017 late bookings) and far apart, which already points to a real difference. I test it formally next.

## 6. Hypothesis test: z-test for two proportions

**Why this test:** the outcome is yes/no (cancelled or not), so I compare two proportions, the same approach as the conversion-rate example in class. The two groups are separate and very large, so the normal approximation holds.

**Assumptions:**
1. Independent bookings. Not fully true, since group bookings can cancel together; I check this with the duplicates test in section 7.
2. At least 10 cancellations and 10 non-cancellations per group (checked below).
3. The data covers two hotels, so the result applies to these hotels.

In [ ]:
# Number of cancellations in each group
x_early = early.sum()
x_late = late.sum()
print("Early: cancelled", x_early, "kept", n_early - x_early)
print("Late: cancelled", x_late, "kept", n_late - x_late)

In [ ]:
# Pooled proportion (under H0 both groups have the same rate)
p_pool = (x_early + x_late) / (n_early + n_late)
p_pool

In [ ]:
# Standard error of the difference
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n_early + 1 / n_late))
se_pool

In [ ]:
# z-statistic
z_stat = (p_early - p_late) / se_pool
z_stat

In [ ]:
# One-sided p-value (right tail, because H1 says early > late)
p_value = 1 - stats.norm.cdf(z_stat)
p_value

In [ ]:
# Python shows 0.0 because the value is too small to display, so I report it as p < 0.001
if p_value < 0.001:
    print("p-value < 0.001")

Python shows 0.0 only because the p-value is too small to display, not because it is exactly zero, so I report it as **p < 0.001**. This is far below 0.05, so I **reject H₀**: early bookings have a significantly higher cancellation rate.

Python shows 0.0 only because the number is too small to display, so I report it as **p < 0.001**, far below 0.05. I **reject H₀**: early bookings have a significantly higher cancellation rate.

In [ ]:
table = pd.crosstab(city["lead_group"], city["is_canceled"])
table

In [ ]:
chi2, p_chi, dof, expected = stats.chi2_contingency(table, correction=False)
print("Chi-square:", round(chi2, 1))
print("z squared:", round(z_stat ** 2, 1))

Both values are about 5,074, so the z-test calculation is correct.

### How big is the difference?
With almost 80,000 bookings, even a tiny gap would be significant, so the size of the gap is what matters.

In [ ]:
# Difference in cancellation rates and its 95% confidence interval
diff = p_early - p_late
se_diff = np.sqrt(p_early * (1 - p_early) / n_early + p_late * (1 - p_late) / n_late)
print("Difference:", round(diff * 100, 1), "percentage points")
print("95% CI:", round((diff - 1.96 * se_diff) * 100, 1), "to", round((diff + 1.96 * se_diff) * 100, 1), "percentage points")

In [ ]:
# Extra cancellations if early bookings cancelled at the late rate
extra_cancellations = (p_early - p_late) * n_early
extra_cancellations

In [ ]:
# Average room revenue of a cancelled early booking (only bookings with a price and at least 1 night)
cancelled_early = city[(city["lead_group"] == "Early") & (city["is_canceled"] == 1)
                       & (city["adr"] > 0) & (city["total_nights"] > 0)]
avg_revenue = cancelled_early["booking_revenue"].mean()
avg_revenue

In [ ]:
print("Revenue linked to the extra cancellations: $", round(extra_cancellations * avg_revenue))

In [ ]:
# Revenue exposure if none, half, or three quarters of the cancelled rooms were resold
exposure = extra_cancellations * avg_revenue
print("No rooms resold: $", round(exposure))
print("Half resold: $", round(exposure * 0.5))
print("Three quarters resold: $", round(exposure * 0.25))

Early bookings cancel about **25 percentage points more** (56% vs 31%), well above my 10-point threshold. That is roughly 8,800 extra cancellations.

I treat the revenue number as an **illustrative gross exposure**, not money the hotel actually lost. It assumes every extra cancellation is worth the average cancelled early booking (about \$315) and that none of those rooms were resold, which gives about \$2.8 million. If half of the rooms were resold, it drops to about \$1.4 million, and if three quarters were resold, to about \$0.7 million. It is also not the revenue a new policy would recover, because the data cannot tell me how many cancellations a policy would actually prevent.

Early bookings cancel about **25 percentage points more** (56% vs 31%), well above my 10-point threshold. That is roughly 8,800 extra cancellations and up to about \$2.8 million of room revenue the hotel held but may not have sold. This is an upper bound, since some cancelled rooms were likely resold. If half of those rooms were resold it would be about \$1.4 million, and if three quarters were resold, about \$0.7 million.

### Does the same thing happen at the Resort Hotel?

In [ ]:
resort = df[df["hotel"] == "Resort Hotel"]
resort.groupby("lead_group")["is_canceled"].mean().round(3)

Early bookings also cancel more at the Resort Hotel (39% vs 20%). This **supports** my result: it is not specific to one hotel.

### Does the result change without the duplicate rows?

In [ ]:
city_no_duplicates = city.drop_duplicates()
city_no_duplicates.groupby("lead_group")["is_canceled"].mean().round(3)

In [ ]:
# Repeat the z-test without duplicate rows
dup_early = city_no_duplicates[city_no_duplicates["lead_group"] == "Early"]["is_canceled"]
dup_late = city_no_duplicates[city_no_duplicates["lead_group"] == "Late"]["is_canceled"]
p_pool_dup = (dup_early.sum() + dup_late.sum()) / (len(dup_early) + len(dup_late))
se_pool_dup = np.sqrt(p_pool_dup * (1 - p_pool_dup) * (1 / len(dup_early) + 1 / len(dup_late)))
z_dup = (dup_early.mean() - dup_late.mean()) / se_pool_dup
print("Early:", len(dup_early), "bookings  Late:", len(dup_late), "bookings")
print("z =", round(z_dup, 2))

In [ ]:
# 95% CI for the difference without duplicates
diff_dup = dup_early.mean() - dup_late.mean()
se_diff_dup = np.sqrt(dup_early.mean() * (1 - dup_early.mean()) / len(dup_early)
                      + dup_late.mean() * (1 - dup_late.mean()) / len(dup_late))
print("Difference:", round(diff_dup * 100, 1), "percentage points")
print("95% CI:", round((diff_dup - 1.96 * se_diff_dup) * 100, 1), "to", round((diff_dup + 1.96 * se_diff_dup) * 100, 1), "percentage points")

`drop_duplicates()` removes rows that are identical in every column. These are not confirmed duplicate bookings, so this is a cautious "worst case" check rather than the correct version of the data.

Without them, early bookings still cancel more (39% vs 25%). The gap is about 14 points (95% CI: 13.2 to 14.9), with z = 33.3 and p < 0.001. Even the bottom of this interval is above my 10-point threshold, so my conclusion holds either way. The true gap is probably somewhere between 14 and 25 points, depending on how many duplicates are real bookings.

Without duplicates the gap shrinks to about 14 points (39% vs 25%, z = 33.3, p < 0.001, 95% CI 13.2 to 14.9 points), because many duplicates are cancelled group bookings. That is still above my 10-point threshold, so my conclusion does not depend on how I treat the duplicates.

In [ ]:
# Share of each market segment in early and late bookings (%)
(pd.crosstab(city["lead_group"], city["market_segment"], normalize="index") * 100).round(1)

In [ ]:
# Cancellation rate for early and late bookings inside the main segments
main_segments = city[city["market_segment"].isin(["Online TA", "Offline TA/TO", "Groups", "Direct"])]
segment_rates = main_segments.groupby(["market_segment", "lead_group"])["is_canceled"].mean().reset_index()
segment_rates["cancellation_pct"] = segment_rates["is_canceled"] * 100
segment_rates

In [ ]:
fig = px.bar(segment_rates, x="market_segment", y="cancellation_pct", color="lead_group", barmode="group",
             text_auto=".0f", title="City Hotel Cancellation Rate by Market Segment",
             labels={"market_segment": "Market segment", "cancellation_pct": "Cancellation rate (%)", "lead_group": "Booking group"})
fig.show()

This **challenges** my result. Early bookings include far more **Groups** (27% vs 10%) and **offline travel agents** (27% vs 16%), which cancel a lot regardless of timing, so part of the gap is about *who* books early. Still, early bookings cancel more within every main segment, including Online TA (46% vs 32%).

### Testing again inside Online TA only
To check that the result is not just groups and agents, I repeat the z-test for Online TA only, the largest segment, which has almost no group contracts.

In [ ]:
ota = city[city["market_segment"] == "Online TA"]
ota_early = ota[ota["lead_group"] == "Early"]["is_canceled"]
ota_late = ota[ota["lead_group"] == "Late"]["is_canceled"]
print("Early:", len(ota_early), "bookings,", round(ota_early.mean(), 3), "cancelled")
print("Late:", len(ota_late), "bookings,", round(ota_late.mean(), 3), "cancelled")

In [ ]:
# Same steps as the main z-test
p_pool_ota = (ota_early.sum() + ota_late.sum()) / (len(ota_early) + len(ota_late))
se_pool_ota = np.sqrt(p_pool_ota * (1 - p_pool_ota) * (1 / len(ota_early) + 1 / len(ota_late)))
z_ota = (ota_early.mean() - ota_late.mean()) / se_pool_ota
p_ota = 1 - stats.norm.cdf(z_ota)
print("z =", round(z_ota, 2), " p-value =", p_ota)

In [ ]:
# 95% CI for the difference inside Online TA
diff_ota = ota_early.mean() - ota_late.mean()
se_diff_ota = np.sqrt(ota_early.mean() * (1 - ota_early.mean()) / len(ota_early)
                      + ota_late.mean() * (1 - ota_late.mean()) / len(ota_late))
print("Difference:", round(diff_ota * 100, 1), "percentage points")
print("95% CI:", round((diff_ota - 1.96 * se_diff_ota) * 100, 1), "to", round((diff_ota + 1.96 * se_diff_ota) * 100, 1), "percentage points")

Inside Online TA the gap is smaller, 13.7 points (95% CI: 12.7 to 14.7), but still significant (z = 27.0, p < 0.001) and above my threshold. This is why my recommendation starts with this segment.

## 8. My interpretation: why do early Online TA bookings cancel more?
The data shows *that* they cancel more, not *why*. My explanation is **"book now, cancel later"**: online travel agencies like Booking.com push free cancellation, so guests can reserve a room months ahead as a free placeholder and cancel if they find something better or their plans change.

If that is right, the data should show two things:
1. Online TA bookings almost never have a deposit, so cancelling costs nothing.
2. Guests who do not engage with their booking (no special requests) cancel much more than guests who do.

In [ ]:
# 1. Deposit type of Online TA bookings (%)
(ota["deposit_type"].value_counts(normalize=True) * 100).round(1)

In [ ]:
# 2. Did the guest make at least one special request?
ota_early_df = ota[ota["lead_group"] == "Early"].copy()
ota_early_df["made_request"] = np.where(ota_early_df["total_of_special_requests"] > 0, "Yes", "No")
ota_early_df.groupby("made_request")["is_canceled"].agg(["count", "mean"]).round(3)

In [ ]:
fig = px.bar(ota_early_df.groupby("made_request")["is_canceled"].mean().reset_index(),
             x="made_request", y="is_canceled", text_auto=".0%",
             title="Early Online TA Bookings: Cancellation Rate by Special Request",
             labels={"made_request": "Guest made a special request", "is_canceled": "Cancellation rate"})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

Both checks fit. 99.9% of Online TA bookings have no deposit, and among early Online TA bookings, guests with **no special request cancelled 66%** of the time vs **33%** for guests who made one. A guest who asks for a quiet room or a crib is planning a real trip; one who never touches the booking looks more like a placeholder.

This is still an association, since guests who make requests may simply be more organised. But it gives the hotel a signal it already has in its system.

**What would prove me wrong:** if cancellation dates showed most cancellations happen close to arrival, or cancelled guests did not rebook elsewhere, that would point more to "plans changed" than to "book now, cancel later".

### A conclusion I cannot make

In [ ]:
# Cancellation rate by deposit type
city.groupby("deposit_type")["is_canceled"].agg(["count", "mean"]).round(3)

In [ ]:
# Who makes the non-refundable bookings?
non_refund = city[city["deposit_type"] == "Non Refund"]
(non_refund["market_segment"].value_counts(normalize=True) * 100).round(1)

In [ ]:
non_refund["lead_time"].median()

Almost 100% of non-refundable bookings were cancelled, which makes it tempting to say deposits do not work, or even cause cancellations. The data does not support that: 97% of these bookings come from Groups and offline travel agents, with a median lead time of 188 days, which are the bookings that cancel most anyway. It also cannot show how guests would react to a new deposit rule.

Likewise, I cannot say that booking early *causes* cancellations; people who book early may simply have less certain plans.

## 9. Recommendation

**What I recommend to the City Hotel Revenue Manager:**
1. **Shorten the free-cancellation window for Online TA bookings made more than 90 days ahead**, for example free cancellation only until 30 days before arrival. I prefer this to a deposit: it targets free placeholders directly without scaring off guests with real plans.
2. **Use special requests as an early warning.** Early Online TA bookings with no request cancel twice as often, so the hotel can send them a reconfirmation email about 45 days before arrival and treat them as higher risk when setting overbooking levels.
3. **Manage groups and offline agents through their contracts** (release dates), since their cancellations are not only about lead time.

**How to measure it:** run the new window on some arrival dates for 8 to 12 weeks and compare with dates on the normal rule. Success means the early Online TA cancellation rate drops by at least 5 points (from about 46%) while early Online TA bookings fall by no more than 5%.

**Evidence:** early bookings cancel 56% vs 31% (z-test, p < 0.001, confirmed with chi-square), the gap grows with lead time, it also appears at the Resort Hotel, and it stays significant without duplicates (14 points) and inside Online TA (46% vs 32%). Within early Online TA bookings, no special request means 66% vs 33% cancelled.

**Why it matters:** early bookings are 44% of City Hotel bookings, and the extra cancellations represent up to about \$2.8 million in room revenue.

**Main risk:** a shorter window could push some guests to competitors that still offer free cancellation, and this data cannot tell me how many.

**What would help:** the pilot results (cancellation rate, bookings and revenue per available room), the date of each cancellation, and whether cancelled guests rebooked at a lower price.

## 10. Limitations
- The data comes from only two hotels and has no booking ID, so I cannot be sure which duplicate rows are real bookings.
- It does not say when the booking was cancelled or if the room was sold again, so the revenue number is a maximum.
- It is observational data, so it shows an association between lead time and cancellations, not a cause.

## AI Use Declaration

- **AI tools used:** Claude (Anthropic).
- **What I used them for:** I used Claude as an assistant throughout the project. It helped me brainstorm research questions, draft and debug the Python code, fix errors in Colab, and tighten my writing. I decided on the business question and the stakeholder, asked for the code to follow the simpler step-by-step style from class, and chose which analyses to keep and which to cut.
- **One AI suggestion I accepted and why:** using a z-test for two proportions to compare the cancellation rates. My outcome is a yes/no variable (cancelled or not) and I am comparing two separate groups, so it was the right test from class for my question.
- **One AI suggestion I rejected or changed and why:** in the confidence interval section, Claude wrote out the full formula for a proportion's confidence interval as an equation. I didn't like that, because I wouldn't write out a full equation in my own explanation, and the code already shows the calculation. I replaced it with a plain sentence: "I build it the same way as in class: the sample proportion, plus or minus 1.96 standard errors." I did the same for the chi-square check, where Claude had written the result as an equation, and I changed it to a normal sentence instead.
- **How I checked the AI's work:** I re-ran the notebook from start to finish, followed the z-test step by step and confirmed it with a chi-square test, and compared every number in my executive summary with the notebook output.